Le fichier adult.data dans le dossier database contient le jeu de données "Adult" (aussi appelé "Census Income") provenant de l'UCI Machine Learning Repository. Ce jeu de données est utilisé pour des tâches de classification, notamment pour prédire si une personne gagne plus ou moins de 50 000 $ par an, à partir de données issues du recensement américain.

À quoi sert ce jeu de données ?

Il est principalement utilisé pour l'apprentissage automatique, l'analyse de données et la pratique de modèles de classification supervisée.
Quelles sont les colonnes ? Chaque ligne représente une personne, avec des attributs comme :

age (âge)
workclass (type d’emploi)
fnlwgt (poids d’échantillon)
education (niveau d’éducation)
education-num (niveau d’éducation numérique)
marital-status (statut marital)
occupation (profession)
relationship (relation familiale)
race (race)
sex (sexe)
capital-gain (gain en capital)
capital-loss (perte en capital)
hours-per-week (heures travaillées par semaine)
native-country (pays d’origine)
income (cible à prédire : >50K ou <=50K)

# ((AAGE > 16) && (AGI > 100) && (AFNLWGT > 1) && (HRSWK > 0))

Voici l’explication précise des conditions d’extraction :

((AAGE > 16) && (AGI > 100) && (AFNLWGT > 1) && (HRSWK > 0))

- AAGE > 16 : On ne garde que les personnes de plus de 16 ans (âge minimum pour travailler aux États-Unis).
- AGI > 100 : Le revenu brut ajusté (Adjusted Gross Income) doit être supérieur à 100 $, pour éliminer les cas aberrants ou sans revenu significatif.
- AFNLWGT > 1 : Le poids d’échantillonnage (final weight, utilisé pour estimer la représentativité dans la population) doit être supérieur à 1, pour éviter les individus qui ne représenteraient quasiment personne.
- HRSWK > 0 : Le nombre d’heures travaillées par semaine doit être strictement positif, donc on exclut les personnes ne travaillant pas du tout.

En résumé : ces filtres permettent de ne garder que les adultes actifs, avec un revenu et une représentativité significatifs, pour que la prédiction du salaire soit pertinente.


AFNLWGT (ou fnlwgt dans le fichier) signifie "final weight" : c’est un poids attribué à chaque individu dans l’échantillon pour que les statistiques calculées soient représentatives de la population américaine réelle.

Plus précisément :

Le recensement ne collecte pas les données de toute la population, mais d’un échantillon.
Chaque personne se voit attribuer un poids (fnlwgt) qui indique combien de personnes similaires (même âge, sexe, origine, etc.) elle représente dans la population totale.
Ces poids sont calculés par le Census Bureau en utilisant plusieurs contrôles statistiques (par état, origine hispanique, âge, sexe, race…).
Lorsqu’on fait des analyses (par exemple, calculer la proportion de personnes gagnant >50K), il faut tenir compte de ce poids pour obtenir des résultats fiables et extrapolables à l’ensemble de la population.
En résumé : fnlwgt permet de corriger le biais d’échantillonnage et d’obtenir des estimations valides pour l’ensemble des États-Unis.

In [2]:
import pandas as pd
df = pd.read_parquet("./adult_all.parquet")
print(df.head())

   age         workclass  education      marital_status         occupation  \
0   39         State-gov  Bachelors       Never-married       Adm-clerical   
1   50  Self-emp-not-inc  Bachelors  Married-civ-spouse    Exec-managerial   
2   38           Private    HS-grad            Divorced  Handlers-cleaners   
3   53           Private       11th  Married-civ-spouse  Handlers-cleaners   
4   28           Private  Bachelors  Married-civ-spouse     Prof-specialty   

   capital_gain  capital_loss  hours_per_week native_country class_label  
0          2174             0              40  United-States       <=50K  
1             0             0              13  United-States       <=50K  
2             0             0              40  United-States       <=50K  
3             0             0              40  United-States       <=50K  
4             0             0              40           Cuba       <=50K  


## Analyse du poids des modèles sauvegardés (ML vs DL)

Le poids d'un modèle sauvegardé dépend de sa structure interne :

- **Modèle ML (RandomForest, etc.)** : stocke la structure des arbres (splits, valeurs), généralement compact (quelques Mo) pour des données tabulaires comme "Adult".
- **Modèle DL (MLP Keras/TensorFlow)** : stocke tous les poids des couches (matrices de paramètres), souvent plus volumineux (plusieurs à dizaines de Mo), même pour un modèle simple.

**Pourquoi cette différence ?**
- Un modèle ML utilise des arbres de décision, qui sont efficaces pour stocker des règles et des splits, donc le fichier reste léger.
- Un modèle DL stocke des matrices de poids pour chaque neurone et chaque couche, ce qui augmente rapidement la taille du fichier, même si le modèle n'est pas plus performant.

**Conséquence :**
- Le modèle ML est plus léger, plus rapide à charger et à déployer.
- Le modèle DL est plus lourd, consomme plus de ressources, et n'apporte pas de gain ici.

Voyons la taille réelle des fichiers modèles :

In [4]:
import os

# Chemins des modèles (à adapter si besoin)
ml_model_path = "../../backend/models/model_rf.pkl"
dl_model_path = "../../backend/models/model_dl.h5"

ml_size = os.path.getsize(ml_model_path) / (1024 * 1024)  # Mo
dl_size = os.path.getsize(dl_model_path) / (1024 * 1024)  # Mo

print(f"Poids du modèle ML (RandomForest): {ml_size:.2f} Mo")
print(f"Poids du modèle DL (MLP Keras): {dl_size:.2f} Mo")

Poids du modèle ML (RandomForest): 95.86 Mo
Poids du modèle DL (MLP Keras): 0.12 Mo
